# 📖 Notebook 1: Metrics Collection & Time-Series Storage

Before we build dashboards or alerts, we need to understand **what metrics are**, **how they're collected**, and **how time-series databases store them**.

## Learning Objectives

By the end of this notebook, you'll understand:
- What a metric is (name, labels, value, timestamp)
- The difference between counters, gauges, and histograms
- How Prometheus scrapes metrics from HTTP endpoints (pull model)
- How time-series data is stored and why specialized databases exist
- How to query metrics using PromQL

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/metrics-monitoring
docker-compose up -d
```

### Visualization Tools

- **Prometheus**: http://localhost:9090 — Run PromQL queries, check scrape targets
- **Grafana**: http://localhost:3000 — Login: admin / admin
- **Adminer** (PostgreSQL): http://localhost:8080 — Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `metrics_demo`
- **RedisInsight** (Redis): http://localhost:5540 — Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import requests
import time
import json
import threading
import random
import math
from http.server import HTTPServer, BaseHTTPRequestHandler

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "metrics_demo",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

PROMETHEUS_URL = "http://localhost:9090"

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker-compose up -d")

try:
    resp = requests.get(f"{PROMETHEUS_URL}/-/healthy", timeout=5)
    print("✅ Connected to Prometheus")
except Exception as e:
    print(f"❌ Prometheus failed: {e}")
    print("   Run: docker-compose up -d")

## 🤔 What Is a Metric?

A **metric** is a numerical measurement taken at a point in time. Every metric has four parts:

```
cpu_usage{host="server-1", region="us-east"} = 0.75   @ 1640000000
  ↑ name        ↑ labels                       ↑ value   ↑ timestamp
```

- **Name**: What you're measuring (e.g., `cpu_usage`, `http_requests_total`)
- **Labels**: Key-value pairs that identify WHERE the metric came from (e.g., `host="server-1"`)
- **Value**: The actual number (e.g., `0.75` = 75% CPU)
- **Timestamp**: When it was measured (Unix epoch seconds)

### What Is a Series?

A **series** is one unique combination of metric name + labels, tracked over time:

```
cpu_usage{host="server-1"} = [0.75, 0.80, 0.72, 0.85, ...]
cpu_usage{host="server-2"} = [0.60, 0.55, 0.62, 0.58, ...]
```

If you have 500,000 servers each reporting `cpu_usage`, that's **500,000 separate series**.  
This is why **cardinality** (the number of unique series) is the central scaling challenge.

In [ ]:
# Let's see what a metric data point looks like in Python

metric_data_point = {
    "name": "cpu_usage",
    "labels": {"host": "server-1", "region": "us-east"},
    "value": 0.75,
    "timestamp": int(time.time())
}

print("📊 A single metric data point:")
print(json.dumps(metric_data_point, indent=2))
print()

# At scale, each data point is tiny (~100-200 bytes)
# But at 5 million per second, that's ~1 GB/s of raw ingestion!
point_size = len(json.dumps(metric_data_point).encode())
print(f"📏 Size of one data point: {point_size} bytes")
print(f"   At 5M points/sec: {point_size * 5_000_000 / 1_000_000_000:.1f} GB/sec")
print()

# Series = unique combo of name + labels
# Each host creates its own series
hosts = ["server-1", "server-2", "server-3"]
metrics = ["cpu_usage", "memory_usage", "disk_usage"]
series_count = len(hosts) * len(metrics)
print(f"🔢 {len(hosts)} hosts × {len(metrics)} metrics = {series_count} unique series")
print(f"   500,000 hosts × 100 metrics = {500_000 * 100:,} series (production scale!)")

## 📊 Metric Types

Not all metrics work the same way. There are three main types:

| Type | Description | Example | Key Property |
|------|-------------|---------|-------------|
| **Gauge** | A value that goes up and down | CPU usage, memory %, temperature | Can be any number |
| **Counter** | A value that only goes up | Total requests, total errors, bytes sent | Monotonically increasing |
| **Histogram** | Distribution of values in buckets | Request latency, response sizes | Shows percentiles (p50, p99) |

Understanding the type matters because it determines **how you query**:
- Gauge → use `avg()`, `max()`, `min()` directly
- Counter → use `rate()` to get per-second change (raw value always goes up)
- Histogram → use `histogram_quantile()` for percentiles

In [ ]:
# Let's simulate all three metric types

print("📊 Gauge: CPU Usage (goes up and down)")
print("=" * 50)
gauge_values = []
cpu = 50.0
for i in range(10):
    cpu += random.uniform(-10, 10)  # fluctuates randomly
    cpu = max(0, min(100, cpu))     # clamp between 0-100
    gauge_values.append(round(cpu, 1))
print(f"  Values: {gauge_values}")
print(f"  Notice: values go UP and DOWN — that's a gauge!")
print()

print("📊 Counter: Total HTTP Requests (only goes up)")
print("=" * 50)
counter_values = []
total = 0
for i in range(10):
    total += random.randint(50, 200)  # new requests each interval
    counter_values.append(total)
print(f"  Values: {counter_values}")
print(f"  Notice: values ONLY GO UP — that's a counter!")
print(f"  To get 'requests per second', use rate(): {counter_values[-1] - counter_values[0]} total over 10 intervals")
print()

print("📊 Histogram: Request Latency Distribution")
print("=" * 50)
latencies = [random.expovariate(1/0.5) for _ in range(100)]  # most fast, some slow
latencies.sort()
p50 = latencies[49]
p90 = latencies[89]
p99 = latencies[98]
print(f"  p50 (median): {p50:.3f}s — half of requests are faster than this")
print(f"  p90:          {p90:.3f}s — 90% of requests are faster")
print(f"  p99:          {p99:.3f}s — 99% of requests are faster")
print(f"  💡 p99 is often 5-10× the median — this is why we monitor percentiles!")

## 🔄 How Prometheus Collects Metrics (Pull Model)

Prometheus uses a **pull model**: it scrapes (fetches) metrics from HTTP endpoints at regular intervals.

```
┌──────────┐    GET /metrics     ┌─────────────┐
│Prometheus │ ──────────────────► │ Your App    │
│ (scraper) │ ◄────────────────── │ (port 8000) │
│           │   text/plain        │             │
└──────────┘   metric lines       └─────────────┘
     │
     │ stores
     ▼
┌──────────┐
│ TSDB     │  (Time-Series Database built into Prometheus)
└──────────┘
```

**How it works:**
1. Your app exposes a `/metrics` HTTP endpoint
2. Prometheus fetches that endpoint every N seconds (configured in `prometheus.yml`)
3. The response is plain text in a specific format
4. Prometheus parses the text and stores each metric as a time-series

**Why pull and not push?**
- Prometheus controls the pace (no overwhelming the server)
- Easy to tell if a target is down (scrape fails)
- No need for the app to know where to send data

Let's build a `/metrics` endpoint and watch Prometheus scrape it!

In [ ]:
# Build a simple metrics endpoint that Prometheus can scrape.
# This simulates a server reporting CPU, memory, latency, etc.

from prometheus_client import (
    Gauge, Counter, Histogram, generate_latest, CONTENT_TYPE_LATEST,
    CollectorRegistry
)

# Create a fresh registry (avoids conflicts if you re-run this cell)
registry = CollectorRegistry()

# Define our metrics — these are the "instruments" that produce data
cpu_gauge = Gauge('demo_cpu_usage', 'CPU usage percentage',
                  ['host', 'region'], registry=registry)
memory_gauge = Gauge('demo_memory_usage', 'Memory usage percentage',
                     ['host', 'region'], registry=registry)
request_latency = Gauge('demo_request_latency', 'Average request latency in seconds',
                        ['host'], registry=registry)
error_rate = Gauge('demo_error_rate', 'Error rate percentage',
                   ['host'], registry=registry)
http_requests = Counter('demo_http_requests_total', 'Total HTTP requests',
                        ['host', 'status'], registry=registry)

# Simulated servers
SERVERS = [
    {"host": "web-1", "region": "us-east"},
    {"host": "web-2", "region": "us-east"},
    {"host": "web-3", "region": "us-west"},
    {"host": "api-1", "region": "us-east"},
    {"host": "api-2", "region": "eu-west"},
]

# Simulate realistic metric values that change over time
sim_tick = 0

def update_metrics():
    """Generate realistic-looking metric values."""
    global sim_tick
    sim_tick += 1
    for server in SERVERS:
        h, r = server["host"], server["region"]
        # CPU: sine wave + noise (simulates daily patterns)
        base_cpu = 40 + 25 * math.sin(sim_tick / 20)
        cpu_val = base_cpu + random.uniform(-10, 10)
        cpu_gauge.labels(host=h, region=r).set(max(0, min(100, cpu_val)))

        # Memory: slowly creeping up (simulates a memory leak!)
        mem_val = 50 + (sim_tick * 0.3) % 40 + random.uniform(-5, 5)
        memory_gauge.labels(host=h, region=r).set(max(0, min(100, mem_val)))

        # Latency: usually low, occasional spikes
        latency = 0.1 + random.expovariate(5)
        if random.random() < 0.05:  # 5% chance of spike
            latency += random.uniform(1, 5)
        request_latency.labels(host=h).set(round(latency, 3))

        # Error rate: mostly low, sometimes spikes
        err = random.expovariate(2)
        if random.random() < 0.03:  # 3% chance of error spike
            err += random.uniform(5, 15)
        error_rate.labels(host=h).set(round(max(0, err), 2))

        # HTTP requests: always incrementing (counter)
        http_requests.labels(host=h, status="200").inc(random.randint(50, 200))
        http_requests.labels(host=h, status="500").inc(random.randint(0, 5))

print("✅ Metric definitions created")
print(f"   Simulating {len(SERVERS)} servers: {[s['host'] for s in SERVERS]}")

In [ ]:
# Start an HTTP server that serves /metrics for Prometheus to scrape.
# This runs in the background so you can keep using the notebook.

class MetricsHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/metrics":
            update_metrics()  # generate fresh values on each scrape
            output = generate_latest(registry)
            self.send_response(200)
            self.send_header("Content-Type", CONTENT_TYPE_LATEST)
            self.end_headers()
            self.wfile.write(output)
        else:
            self.send_response(404)
            self.end_headers()

    def log_message(self, format, *args):
        pass  # suppress request logs

# Start server in background thread
server = HTTPServer(("0.0.0.0", 8000), MetricsHandler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()

print("✅ Metrics server running on http://localhost:8000/metrics")
print("   Prometheus is configured to scrape this every 5 seconds.")
print()
print("👀 Try it yourself:")
print("   1. Open http://localhost:8000/metrics in your browser")
print("   2. Open Prometheus: http://localhost:9090/targets")
print("      You should see 'demo_app' target with state UP")

In [ ]:
# Let's see what the /metrics endpoint looks like — this is exactly
# what Prometheus sees when it scrapes our app.

resp = requests.get("http://localhost:8000/metrics")
lines = resp.text.strip().split("\n")

print("📄 Raw /metrics output (first 25 lines):")
print("=" * 70)
for line in lines[:25]:
    print(line)
print("...")
print()
print(f"Total lines: {len(lines)}")
print()
print("💡 This is plain text! Each line is one metric with labels and a value.")
print("   Lines starting with # are HELP (description) and TYPE (gauge/counter/etc).")

## 💾 Time-Series Storage: Why Not Just Use Postgres?

Let's store the same metrics in both Postgres and see why specialized time-series databases exist.

### The Naive Approach: Postgres

```sql
CREATE TABLE metrics (
    id SERIAL,
    metric_name TEXT,
    labels JSONB,
    value DOUBLE PRECISION,
    timestamp TIMESTAMPTZ
);
```

This works fine for small scale. But at 5M writes/sec:
- Postgres can't sustain that write throughput
- Indexes bloat as data grows
- DELETEs for retention cause vacuum pressure
- Queries over weeks of data are painfully slow

### Time-Series Databases Are Optimized For This

| Feature | Postgres | Time-Series DB (Prometheus/InfluxDB) |
|---------|----------|-------------------------------------|
| Write pattern | Random inserts | Append-only (sequential) |
| Compression | General purpose | 10-20× (timestamps + values compress well) |
| Retention | Manual DELETE + VACUUM | Drop old time chunks instantly |
| Query speed | Scans entire table | Skips irrelevant time ranges |
| Rollups | Manual materialized views | Built-in downsampling |

In [ ]:
# Let's compare: write metrics to Postgres and measure performance

conn = get_db_connection()
cursor = conn.cursor()

# Create a simple metrics table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS metrics_raw (
        id SERIAL PRIMARY KEY,
        metric_name TEXT NOT NULL,
        labels JSONB,
        value DOUBLE PRECISION NOT NULL,
        ts TIMESTAMPTZ NOT NULL DEFAULT NOW()
    )
""")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_metrics_ts ON metrics_raw(ts)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_metrics_name ON metrics_raw(metric_name)")
conn.commit()

# Write 10,000 metric data points (simulating a burst)
start = time.time()
for i in range(10_000):
    host = f"server-{i % 100}"
    cursor.execute(
        "INSERT INTO metrics_raw (metric_name, labels, value, ts) VALUES (%s, %s, %s, NOW())",
        ("cpu_usage", json.dumps({"host": host}), random.uniform(0, 100))
    )
conn.commit()
write_time = time.time() - start

print(f"📝 Wrote 10,000 metrics to Postgres in {write_time:.2f}s")
print(f"   Throughput: {10_000 / write_time:,.0f} writes/sec")
print(f"   We need 5,000,000/sec — Postgres gives us {10_000 / write_time:,.0f}/sec")
print(f"   That's {5_000_000 / (10_000 / write_time):,.0f}× too slow!")
print()
print("💡 This is why production metrics systems use specialized time-series databases.")
print("   They use append-only writes, columnar compression, and time-based partitioning.")

In [ ]:
# Now let's query Postgres and see how it handles time-range queries

# Query 1: Average CPU for all servers in the last minute
start = time.time()
cursor.execute("""
    SELECT 
        labels->>'host' AS host,
        AVG(value) AS avg_cpu,
        COUNT(*) AS samples
    FROM metrics_raw
    WHERE metric_name = 'cpu_usage'
      AND ts >= NOW() - INTERVAL '1 minute'
    GROUP BY labels->>'host'
    ORDER BY avg_cpu DESC
    LIMIT 10
""")
results = cursor.fetchall()
query_time = (time.time() - start) * 1000

print(f"📊 Top 10 hosts by CPU (last 1 minute) — {query_time:.1f}ms")
print(f"{'Host':<15} {'Avg CPU':>10} {'Samples':>10}")
print("-" * 40)
for host, avg_cpu, samples in results:
    print(f"{host:<15} {avg_cpu:>9.1f}% {samples:>10}")

print()
print("💡 This works for 10K rows. But imagine 30 days of data at 5M/sec...")
print(f"   That's {5_000_000 * 86400 * 30:,.0f} rows ({5_000_000 * 86400 * 30 * 100 / 1e12:.1f} TB)")
print("   This query would take minutes, not milliseconds!")

conn.close()

## 🔍 Querying Prometheus with PromQL

Now let's query the **real** time-series database — Prometheus. It has its own query language called **PromQL**.

Wait about 30 seconds after starting the metrics server so Prometheus has time to collect some data.

### Common PromQL Patterns

| Query | What It Does |
|-------|--------------|
| `demo_cpu_usage` | Get the latest value for all series |
| `demo_cpu_usage{host="web-1"}` | Filter by label |
| `avg(demo_cpu_usage)` | Average across all hosts |
| `avg by (region)(demo_cpu_usage)` | Average grouped by region |
| `rate(demo_http_requests_total[1m])` | Requests per second (over 1 min window) |
| `max_over_time(demo_cpu_usage[5m])` | Peak CPU in last 5 minutes |

In [ ]:
# Helper function to query Prometheus

def prom_query(query: str) -> dict:
    """Run a PromQL instant query and return parsed results."""
    resp = requests.get(f"{PROMETHEUS_URL}/api/v1/query", params={"query": query})
    data = resp.json()
    if data["status"] != "success":
        print(f"❌ Query failed: {data.get('error', 'unknown error')}")
        return []
    return data["data"]["result"]

def prom_range_query(query: str, start: float, end: float, step: str = "15s") -> dict:
    """Run a PromQL range query and return parsed results."""
    resp = requests.get(f"{PROMETHEUS_URL}/api/v1/query_range", params={
        "query": query, "start": start, "end": end, "step": step
    })
    data = resp.json()
    if data["status"] != "success":
        print(f"❌ Query failed: {data.get('error', 'unknown error')}")
        return []
    return data["data"]["result"]

# Wait for Prometheus to have some data
print("⏳ Waiting 15 seconds for Prometheus to scrape some data...")
time.sleep(15)

# Query 1: Get latest CPU for all hosts
print("\n📊 Query: demo_cpu_usage (latest values)")
print("=" * 60)
results = prom_query("demo_cpu_usage")
for r in results:
    labels = r["metric"]
    value = float(r["value"][1])
    print(f"  {labels.get('host', '?'):<10} region={labels.get('region', '?'):<10} CPU={value:.1f}%")

print()
print("💡 Each line is a separate SERIES (unique host + region combination)")

In [ ]:
# More PromQL queries — filtering, aggregation, and rates

# Query 2: Average CPU by region
print("📊 Query: avg by (region)(demo_cpu_usage)")
print("=" * 50)
results = prom_query('avg by (region)(demo_cpu_usage)')
for r in results:
    region = r["metric"].get("region", "unknown")
    value = float(r["value"][1])
    print(f"  region={region:<12} avg CPU={value:.1f}%")

print()

# Query 3: HTTP request rate per host
print("📊 Query: rate(demo_http_requests_total[1m])")
print("=" * 50)
results = prom_query('sum by (host)(rate(demo_http_requests_total[1m]))')
for r in results:
    host = r["metric"].get("host", "unknown")
    value = float(r["value"][1])
    print(f"  {host:<12} {value:.1f} req/sec")

print()

# Query 4: Max CPU over the last 2 minutes
print("📊 Query: max_over_time(demo_cpu_usage[2m])")
print("=" * 50)
results = prom_query('max by (host)(max_over_time(demo_cpu_usage[2m]))')
for r in results:
    host = r["metric"].get("host", "unknown")
    value = float(r["value"][1])
    print(f"  {host:<12} peak CPU={value:.1f}%")

print()
print("💡 Notice how PromQL lets you aggregate, filter, and compute over time windows.")
print("   This is much more natural than SQL for time-series data!")

## ⚡ Caching Query Results with Redis

In production, dashboard queries hit the time-series DB repeatedly for the same data.  
Adding a **Redis cache** in front of the query path avoids redundant work.

The key insight: queries separated by 10 seconds cover **almost identical data** — the only difference is the newest 10 seconds. We can cache the historical part and only query the fresh data.

```
Dashboard → Query Service → Redis Cache (hit?) → Prometheus (miss only)
```

In [ ]:
# Demonstrate caching Prometheus query results in Redis

r = get_redis_client()

def cached_prom_query(query: str, ttl_seconds: int = 15) -> list:
    """
    Query Prometheus with a Redis cache layer.
    Identical queries within the TTL window return cached results.
    """
    import hashlib
    cache_key = f"prom_cache:{hashlib.md5(query.encode()).hexdigest()}"

    # Check cache first
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), True  # cache HIT

    # Cache miss — query Prometheus
    results = prom_query(query)

    # Store in Redis with TTL
    r.setex(cache_key, ttl_seconds, json.dumps(results, default=str))

    return results, False  # cache MISS

# First query: cache MISS (fetches from Prometheus)
start = time.time()
results, hit = cached_prom_query("avg by (region)(demo_cpu_usage)")
t1 = (time.time() - start) * 1000
print(f"Query 1: {'🟢 HIT' if hit else '🔴 MISS'} — {t1:.1f}ms")

# Second query: cache HIT (served from Redis)
start = time.time()
results, hit = cached_prom_query("avg by (region)(demo_cpu_usage)")
t2 = (time.time() - start) * 1000
print(f"Query 2: {'🟢 HIT' if hit else '🔴 MISS'} — {t2:.1f}ms")

# Third query: still cached
start = time.time()
results, hit = cached_prom_query("avg by (region)(demo_cpu_usage)")
t3 = (time.time() - start) * 1000
print(f"Query 3: {'🟢 HIT' if hit else '🔴 MISS'} — {t3:.1f}ms")

print(f"\n📊 Speedup: {t1/t2:.1f}× faster with cache!")
print()
print("💡 In production, a busy dashboard with 10 panels refreshing every 5s")
print("   would generate 120 queries/minute. Caching reduces DB load dramatically.")
print()
print("👀 Open RedisInsight (http://localhost:5540) to see the cached keys!")

## 📉 Rollups: Serving Long-Range Queries Fast

Querying raw 10-second data for a 30-day dashboard is brutal:
- 30 days × 86,400 sec/day ÷ 10 = **259,200 data points per series**
- × 1,000 servers = **259 million rows** for one panel!

The solution: **pre-compute aggregates** at coarser resolutions.

| Resolution | Retention | Points per series (30 days) |
|-----------|-----------|----------------------------|
| Raw (10s) | 2 days | 17,280 |
| 1-minute | 2 weeks | 43,200 |
| 1-hour | 90 days | 720 |
| 1-day | 2 years | 30 |

A 30-day query uses **hourly rollups** (720 points) instead of raw data (259,200 points).  
That's **360× less data** to scan!

In [ ]:
# Demonstrate rollups: compute 1-minute aggregates from raw data

conn = get_db_connection()
cursor = conn.cursor()

# Create a rollup table for 1-minute aggregates
cursor.execute("""
    CREATE TABLE IF NOT EXISTS metrics_1m_rollup (
        metric_name TEXT NOT NULL,
        host TEXT NOT NULL,
        bucket TIMESTAMPTZ NOT NULL,
        avg_value DOUBLE PRECISION,
        min_value DOUBLE PRECISION,
        max_value DOUBLE PRECISION,
        sample_count INTEGER,
        PRIMARY KEY (metric_name, host, bucket)
    )
""")
conn.commit()

# Compute the rollup from raw data
cursor.execute("""
    INSERT INTO metrics_1m_rollup (metric_name, host, bucket, avg_value, min_value, max_value, sample_count)
    SELECT
        metric_name,
        labels->>'host' AS host,
        date_trunc('minute', ts) AS bucket,
        AVG(value),
        MIN(value),
        MAX(value),
        COUNT(*)
    FROM metrics_raw
    WHERE metric_name = 'cpu_usage'
    GROUP BY metric_name, labels->>'host', date_trunc('minute', ts)
    ON CONFLICT (metric_name, host, bucket) DO UPDATE SET
        avg_value = EXCLUDED.avg_value,
        min_value = EXCLUDED.min_value,
        max_value = EXCLUDED.max_value,
        sample_count = EXCLUDED.sample_count
""")
conn.commit()

# Compare sizes
cursor.execute("SELECT COUNT(*) FROM metrics_raw WHERE metric_name = 'cpu_usage'")
raw_count = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM metrics_1m_rollup")
rollup_count = cursor.fetchone()[0]

print(f"📊 Raw data points:    {raw_count:,}")
print(f"📊 1-minute rollups:   {rollup_count:,}")
if rollup_count > 0:
    print(f"📊 Compression ratio:  {raw_count / rollup_count:.0f}×")

print()

# Show sample rollup data
cursor.execute("""
    SELECT host, bucket, round(avg_value::numeric, 1) as avg_cpu,
           round(min_value::numeric, 1) as min_cpu,
           round(max_value::numeric, 1) as max_cpu,
           sample_count
    FROM metrics_1m_rollup
    ORDER BY bucket DESC, host
    LIMIT 10
""")
print(f"{'Host':<12} {'Bucket':<22} {'Avg':>6} {'Min':>6} {'Max':>6} {'N':>4}")
print("-" * 65)
for row in cursor.fetchall():
    print(f"{row[0]:<12} {str(row[1]):<22} {row[2]:>6} {row[3]:>6} {row[4]:>6} {row[5]:>4}")

print()
print("💡 Rollups are LOSSY — you can't recover individual 10s values from the avg.")
print("   But for dashboards showing 30-day trends, you don't need that detail!")

conn.close()

## 📚 Summary

### Key Takeaways

1. **Metrics** have four parts: name, labels, value, timestamp. A **series** is one unique name+labels combo over time.
2. **Metric types** matter: gauges fluctuate, counters only go up (use `rate()`), histograms show distributions.
3. **Prometheus pulls** metrics from HTTP endpoints — your app exposes `/metrics` and Prometheus scrapes it.
4. **Postgres can't handle** 5M writes/sec — time-series databases use append-only writes and time-based partitioning.
5. **Redis caching** speeds up dashboard queries by avoiding redundant DB hits.
6. **Rollups** pre-compute aggregates so 30-day queries scan 720 points instead of 259,200.

### Next Up

In **Notebook 2**, we'll build **alerting rules and thresholds** — how to define conditions that trigger notifications when your system is in trouble.